# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
import toolbox_ML as tool
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv", index_col= "laptop_ID")

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [4]:
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [6]:
df.describe()

,Inches,Price_in_euros
count,912.000000,912.000000
mean,14.981579,1111.724090
std,1.436719,687.959172
min,10.100000,174.000000
25%,14.000000,589.000000
50%,15.600000,978.000000
75%,15.600000,1483.942500
max,18.400000,6099.000000


In [7]:
df.columns = df.columns.str.lower()
print(df.columns)

Index(['company', 'product', 'typename', 'inches', 'screenresolution', 'cpu',
       'ram', 'memory', 'gpu', 'opsys', 'weight', 'price_in_euros'],
      dtype='object')


### 2.3 Definir X e y

In [8]:
X = df.drop(['price_in_euros'], axis=1)
y = df['price_in_euros'].copy()
X.shape

(912, 11)

In [9]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [11]:
X_train

,company,product,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight
laptop_ID,,,,,,,,,,,
1118,HP,ZBook 17,Workstation,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,8GB,1TB HDD,AMD FirePro W6150M,Windows 7,3.0kg
153,Dell,Inspiron 5577,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1050,Windows 10,2.56kg
275,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.9GHz,8GB,512GB SSD,Intel Iris Graphics 550,macOS,1.37kg
1100,HP,EliteBook 840,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,500GB HDD,Intel HD Graphics 520,Windows 7,1.54kg
131,Dell,Inspiron 5770,Notebook,17.3,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,16GB,256GB SSD + 2TB HDD,AMD Radeon 530,Windows 10,2.8kg
...,...,...,...,...,...,...,...,...,...,...,...
578,HP,14-am079na (N3710/8GB/2TB/W10),Notebook,14.0,1366x768,Intel Pentium Quad Core N3710 1.6GHz,8GB,2TB HDD,Intel HD Graphics 405,Windows 10,1.94kg
996,Lenovo,IdeaPad 320-15ABR,Notebook,15.6,Full HD 1920x1080,AMD A12-Series 9720P 3.6GHz,6GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg
770,Dell,Latitude 7280,Ultrabook,12.5,Full HD 1920x1080,Intel Core i7 7600U 2.8GHz,16GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.18kg


In [12]:
y_train

laptop_ID
1118    2899.00
153     1249.26
275     1958.90
1100    1030.99
131     1396.00
         ...   
578      389.00
996      549.00
770     1859.00
407      306.00
418     1943.00
Name: price_in_euros, Length: 729, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [13]:
target = df["price_in_euros"]
target_str = "price_in_euros"

In [14]:
X_train.columns

Index(['company', 'product', 'typename', 'inches', 'screenresolution', 'cpu',
       'ram', 'memory', 'gpu', 'opsys', 'weight'],
      dtype='object')

In [15]:
tool.describe_df(X_train)

,data_type,valores_nulos,valores_unicos,cardinalidad_ptg
company,categorica,0.0,17,2.33
product,categorica,0.0,408,55.97
typename,categorica,0.0,6,0.82
inches,numerica,0.0,15,2.06
screenresolution,categorica,0.0,33,4.53
cpu,categorica,0.0,93,12.76
ram,categorica,0.0,8,1.1
memory,categorica,0.0,33,4.53
gpu,categorica,0.0,84,11.52
opsys,categorica,0.0,9,1.23


In [16]:
print(X_train.ram.value_counts())

ram
8GB     340
4GB     217
16GB    112
6GB      20
12GB     18
2GB      15
32GB      6
64GB      1
Name: count, dtype: int64


In [17]:
X_train["ram_num"]= X_train["ram"].str.replace("GB", "")
X_train["ram_num"] = X_train["ram_num"].astype(int)

In [18]:
X_test["ram_num"]= X_test["ram"].str.replace("GB", "")
X_test["ram_num"] = X_test["ram_num"].astype(int)

In [19]:
X_train["weight"]= X_train["weight"].str.lower()
X_train["weight_num"]= X_train["weight"].str.replace("kg", "")
X_train["weight_num"] = X_train["weight_num"].astype(float)

In [20]:
X_test["weight"]= X_test["weight"].str.lower()
X_test["weight_num"]= X_test["weight"].str.replace("kg", "")
X_test["weight_num"] = X_test["weight_num"].astype(float)

In [21]:
features_quitar = ["ram"]
features_quitar.append("weight")
features_quitar

['ram', 'weight']

In [22]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 729 entries, 1118 to 418
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   company           729 non-null    object 
 1   product           729 non-null    object 
 2   typename          729 non-null    object 
 3   inches            729 non-null    float64
 4   screenresolution  729 non-null    object 
 5   cpu               729 non-null    object 
 6   ram               729 non-null    object 
 7   memory            729 non-null    object 
 8   gpu               729 non-null    object 
 9   opsys             729 non-null    object 
 10  weight            729 non-null    object 
 11  ram_num           729 non-null    int64  
 12  weight_num        729 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 79.7+ KB


In [23]:
print(X_train.typename.value_counts())

typename
Notebook              412
Gaming                113
Ultrabook             113
2 in 1 Convertible     62
Netbook                16
Workstation            13
Name: count, dtype: int64


In [24]:
print(list(X_train.screenresolution.unique()))

['IPS Panel Full HD 1920x1080', 'Full HD 1920x1080', 'IPS Panel Retina Display 2560x1600', '1366x768', 'Quad HD+ / Touchscreen 3200x1800', '1600x900', 'IPS Panel 2560x1440', 'IPS Panel Retina Display 2304x1440', 'IPS Panel 4K Ultra HD / Touchscreen 3840x2160', 'IPS Panel Full HD / Touchscreen 1920x1080', 'IPS Panel Quad HD+ / Touchscreen 3200x1800', 'Full HD / Touchscreen 1920x1080', '1440x900', '1920x1080', 'Touchscreen 2400x1600', '2560x1440', 'IPS Panel Retina Display 2880x1800', 'Touchscreen / Full HD 1920x1080', 'Touchscreen 2560x1440', 'Touchscreen 1366x768', '4K Ultra HD 3840x2160', 'Touchscreen 2256x1504', 'IPS Panel 1366x768', 'Quad HD+ 3200x1800', 'IPS Panel Quad HD+ 2560x1440', 'IPS Panel 4K Ultra HD 3840x2160', '4K Ultra HD / Touchscreen 3840x2160', 'IPS Panel Touchscreen 1920x1200', 'IPS Panel Touchscreen 2560x1440', 'IPS Panel Quad HD+ 3200x1800', 'Touchscreen / Quad HD+ 3200x1800', 'Touchscreen / 4K Ultra HD 3840x2160', 'IPS Panel Full HD 2560x1440']


In [25]:
# X_train.ScreenResolution.isin(["IPS Panel Full HD 2560x1440"]) ---> Dice False cuando no coincide y True cuando si
X_train[X_train.screenresolution == "IPS Panel Full HD 2560x1440"]

,company,product,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,ram_num,weight_num
laptop_ID,,,,,,,,,,,,,
418,Lenovo,Thinkpad T470p,Ultrabook,14.0,IPS Panel Full HD 2560x1440,Intel Core i7 7700HQ 2.8GHz,8GB,512GB SSD,Nvidia GeForce GT 940MX,Windows 10,1.7kg,8,1.7


In [26]:
X_train[X_train.screenresolution == "IPS Panel Quad HD+ 2560x1440"]

,company,product,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,ram_num,weight_num
laptop_ID,,,,,,,,,,,,,
840,Lenovo,Thinkpad X1,Ultrabook,14.0,IPS Panel Quad HD+ 2560x1440,Intel Core i7 6600U 2.6GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.1kg,16,1.10
728,Lenovo,ThinkPad X1,Ultrabook,14.0,IPS Panel Quad HD+ 2560x1440,Intel Core i7 6500U 2.5GHz,8GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.17kg,8,1.17
476,Lenovo,Thinkpad T460s,Ultrabook,14.0,IPS Panel Quad HD+ 2560x1440,Intel Core i7 6600U 2.6GHz,12GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.4kg,12,1.40


In [27]:
X_train.screenresolution = X_train.screenresolution.replace({
    "IPS Panel Full HD 2560x1440"   : "IPS Panel 2560x1440",
    "IPS Panel Quad HD+ 2560x1440"  : "IPS Panel 2560x1440"
})

In [28]:
X_test.screenresolution = X_test.screenresolution.replace({
    "IPS Panel Full HD 2560x1440"   : "IPS Panel 2560x1440",
    "IPS Panel Quad HD+ 2560x1440"  : "IPS Panel 2560x1440"
})

In [29]:
print(list(X_train.screenresolution.unique()))

['IPS Panel Full HD 1920x1080', 'Full HD 1920x1080', 'IPS Panel Retina Display 2560x1600', '1366x768', 'Quad HD+ / Touchscreen 3200x1800', '1600x900', 'IPS Panel 2560x1440', 'IPS Panel Retina Display 2304x1440', 'IPS Panel 4K Ultra HD / Touchscreen 3840x2160', 'IPS Panel Full HD / Touchscreen 1920x1080', 'IPS Panel Quad HD+ / Touchscreen 3200x1800', 'Full HD / Touchscreen 1920x1080', '1440x900', '1920x1080', 'Touchscreen 2400x1600', '2560x1440', 'IPS Panel Retina Display 2880x1800', 'Touchscreen / Full HD 1920x1080', 'Touchscreen 2560x1440', 'Touchscreen 1366x768', '4K Ultra HD 3840x2160', 'Touchscreen 2256x1504', 'IPS Panel 1366x768', 'Quad HD+ 3200x1800', 'IPS Panel 4K Ultra HD 3840x2160', '4K Ultra HD / Touchscreen 3840x2160', 'IPS Panel Touchscreen 1920x1200', 'IPS Panel Touchscreen 2560x1440', 'IPS Panel Quad HD+ 3200x1800', 'Touchscreen / Quad HD+ 3200x1800', 'Touchscreen / 4K Ultra HD 3840x2160']


In [30]:
X_train["screenresolution"] = X_train["screenresolution"].str.lower()
X_test["screenresolution"] = X_test["screenresolution"].str.lower()

In [31]:
print(X_train.cpu.value_counts(),"\n","Valores únicos: ", len(X_train.cpu.unique()))

cpu
Intel Core i5 7200U 2.5GHz       100
Intel Core i7 7700HQ 2.8GHz       82
Intel Core i7 7500U 2.7GHz        77
Intel Core i5 8250U 1.6GHz        46
Intel Core i7 8550U 1.8GHz        40
                                ... 
Intel Core i5 7500U 2.7GHz         1
Intel Core i3 6100U 2.1GHz         1
Intel Core M 1.1GHz                1
Intel Xeon E3-1535M v5 2.9GHz      1
AMD E-Series 6110 1.5GHz           1
Name: count, Length: 93, dtype: int64 
 Valores únicos:  93


In [32]:
# ORDINAL ENCODING
from sklearn.preprocessing import OrdinalEncoder
categories_typename= [["Netbook", "Notebook", "2 in 1 Convertible", "Ultrabook", "Gaming", "Workstation"]]
ordinal_encoder = OrdinalEncoder(categories=categories_typename)
X_train["typename_encoded"] = ordinal_encoder.fit_transform(X_train[["typename"]])
X_test["typename_encoded"] = ordinal_encoder.transform(X_test[["typename"]])


In [33]:
X_train.columns

Index(['company', 'product', 'typename', 'inches', 'screenresolution', 'cpu',
       'ram', 'memory', 'gpu', 'opsys', 'weight', 'ram_num', 'weight_num',
       'typename_encoded'],
      dtype='object')

In [34]:
features_quitar.append("typename")
features_quitar

['ram', 'weight', 'typename']

In [35]:
mapeo = {
    # grupo 1
    '1366x768':0,
    'touchscreen 1366x768':0,
    'ips panel 1366x768':0,
    'ips panel touchscreen 1366x768':0,
    # grupo 2
    '1440x900':1,
    # grupo 3
    '1600x900':2,
    # grupo 4
    '1920x1080':3,
    'full hd 1920x1080':3,
    'full hd / touchscreen 1920x1080':3,
    'touchscreen / full hd 1920x1080':3,
    'ips panel full hd 1920x1080':3,
    'ips panel full hd / touchscreen 1920x1080':3,
    'ips panel touchscreen 1920x1200':3,
    'ips panel full hd 1920x1200':3,
    # grupo 5
    '2560x1440':4,
    'touchscreen 2560x1440':4,
    'ips panel 2560x1440':4,
    'ips panel touchscreen 2560x1440':4,
    'ips panel full hd 2160x1440':4,
    # grupo 6
    'ips panel retina display 2304x1440':5,
    'touchscreen 2256x1504':5,
    'touchscreen 2400x1600':5,
    'ips panel retina display 2560x1600':5,
    'ips panel retina display 2880x1800':5,
    # grupo 7
    'quad hd+ 3200x1800':6,
    'quad hd+ / touchscreen 3200x1800':6,
    'touchscreen / quad hd+ 3200x1800':6,
    'ips panel quad hd+ / touchscreen 3200x1800':6,
    'ips panel quad hd+ 3200x1800':6,
    # grupo 8
    '4k ultra hd 3840x2160':7,
    '4k ultra hd / touchscreen 3840x2160':7,
    'touchscreen / 4k ultra hd 3840x2160':7,
    'ips panel 4k ultra hd 3840x2160':7,
    'ips panel 4k ultra hd / touchscreen 3840x2160':7
}

X_train["screenresolution_encoded"] = X_train["screenresolution"].map(mapeo)
X_test["screenresolution_encoded"] = X_test["screenresolution"].map(mapeo)

In [36]:
X_train["screenresolution_encoded"].value_counts()

screenresolution_encoded
3    465
0    179
7     21
6     20
2     14
5     13
4     13
1      4
Name: count, dtype: int64

In [37]:
features_quitar.append("screenresolution")
features_quitar

['ram', 'weight', 'typename', 'screenresolution']

In [38]:
X_train.columns

Index(['company', 'product', 'typename', 'inches', 'screenresolution', 'cpu',
       'ram', 'memory', 'gpu', 'opsys', 'weight', 'ram_num', 'weight_num',
       'typename_encoded', 'screenresolution_encoded'],
      dtype='object')

In [39]:
X_train.head()

,company,product,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,ram_num,weight_num,typename_encoded,screenresolution_encoded
laptop_ID,,,,,,,,,,,,,,,
1118,HP,ZBook 17,Workstation,17.3,ips panel full hd 1920x1080,Intel Core i7 6700HQ 2.6GHz,8GB,1TB HDD,AMD FirePro W6150M,Windows 7,3.0kg,8,3.00,5.0,3
153,Dell,Inspiron 5577,Gaming,15.6,full hd 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1050,Windows 10,2.56kg,16,2.56,4.0,3
275,Apple,MacBook Pro,Ultrabook,13.3,ips panel retina display 2560x1600,Intel Core i5 2.9GHz,8GB,512GB SSD,Intel Iris Graphics 550,macOS,1.37kg,8,1.37,3.0,5
1100,HP,EliteBook 840,Notebook,14.0,full hd 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,500GB HDD,Intel HD Graphics 520,Windows 7,1.54kg,4,1.54,1.0,3
131,Dell,Inspiron 5770,Notebook,17.3,full hd 1920x1080,Intel Core i7 8550U 1.8GHz,16GB,256GB SSD + 2TB HDD,AMD Radeon 530,Windows 10,2.8kg,16,2.80,1.0,3


In [40]:
print(tool.describe_df(X_train))

                           data_type valores_nulos valores_unicos  \
company                   categorica           0.0             17   
product                   categorica           0.0            408   
typename                  categorica           0.0              6   
inches                      numerica           0.0             15   
screenresolution          categorica           0.0             31   
cpu                       categorica           0.0             93   
ram                       categorica           0.0              8   
memory                    categorica           0.0             33   
gpu                       categorica           0.0             84   
opsys                     categorica           0.0              9   
weight                    categorica           0.0            148   
ram_num                     numerica           0.0              8   
weight_num                  numerica           0.0            142   
typename_encoded            numeri

In [41]:
from sklearn.preprocessing import OneHotEncoder
encoder_company = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# Train
encoded_train = encoder_company.fit_transform(X_train[['company']])
encoded_train_df = pd.DataFrame(encoded_train, columns=encoder_company.get_feature_names_out(), index=X_train.index)
X_train = pd.concat([X_train, encoded_train_df], axis=1)

# Test
encoded_test = encoder_company.transform(X_test[['company']])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder_company.get_feature_names_out(), index=X_test.index)
X_test = pd.concat([X_test, encoded_test_df], axis=1)


c:\Users\Arrate\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [42]:
X_train.columns

Index(['company', 'product', 'typename', 'inches', 'screenresolution', 'cpu',
       'ram', 'memory', 'gpu', 'opsys', 'weight', 'ram_num', 'weight_num',
       'typename_encoded', 'screenresolution_encoded', 'company_Apple',
       'company_Asus', 'company_Chuwi', 'company_Dell', 'company_Fujitsu',
       'company_Google', 'company_HP', 'company_Lenovo', 'company_MSI',
       'company_Mediacom', 'company_Microsoft', 'company_Razer',
       'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi'],
      dtype='object')

In [43]:
features_quitar.append("company")
features_quitar

['ram', 'weight', 'typename', 'screenresolution', 'company']

In [44]:
X_train = X_train.drop(columns=features_quitar)
X_test = X_test.drop(columns=features_quitar)

X_train.columns

Index(['product', 'inches', 'cpu', 'memory', 'gpu', 'opsys', 'ram_num',
       'weight_num', 'typename_encoded', 'screenresolution_encoded',
       'company_Apple', 'company_Asus', 'company_Chuwi', 'company_Dell',
       'company_Fujitsu', 'company_Google', 'company_HP', 'company_Lenovo',
       'company_MSI', 'company_Mediacom', 'company_Microsoft', 'company_Razer',
       'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi'],
      dtype='object')

In [45]:
encoder_opsys = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# Train
encoded_train = encoder_opsys.fit_transform(X_train[['opsys']])
encoded_train_df = pd.DataFrame(encoded_train, columns=encoder_opsys.get_feature_names_out(), index=X_train.index)
X_train = pd.concat([X_train, encoded_train_df], axis=1)

# Test
encoded_test = encoder_opsys.transform(X_test[['opsys']])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder_opsys.get_feature_names_out(), index=X_test.index)
X_test = pd.concat([X_test, encoded_test_df], axis=1)


In [46]:
X_train = X_train.drop(columns=['opsys'])
X_test = X_test.drop(columns=['opsys'])
X_train.columns

Index(['product', 'inches', 'cpu', 'memory', 'gpu', 'ram_num', 'weight_num',
       'typename_encoded', 'screenresolution_encoded', 'company_Apple',
       'company_Asus', 'company_Chuwi', 'company_Dell', 'company_Fujitsu',
       'company_Google', 'company_HP', 'company_Lenovo', 'company_MSI',
       'company_Mediacom', 'company_Microsoft', 'company_Razer',
       'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi',
       'opsys_Chrome OS', 'opsys_Linux', 'opsys_Mac OS X', 'opsys_No OS',
       'opsys_Windows 10', 'opsys_Windows 10 S', 'opsys_Windows 7',
       'opsys_macOS'],
      dtype='object')

In [47]:
print(X_train[['product', 'cpu', 'memory', 'gpu']].head())

                 product                          cpu                memory  \
laptop_ID                                                                     
1118            ZBook 17  Intel Core i7 6700HQ 2.6GHz               1TB HDD   
153        Inspiron 5577  Intel Core i7 7700HQ 2.8GHz             512GB SSD   
275          MacBook Pro         Intel Core i5 2.9GHz             512GB SSD   
1100       EliteBook 840   Intel Core i5 6200U 2.3GHz             500GB HDD   
131        Inspiron 5770   Intel Core i7 8550U 1.8GHz  256GB SSD +  2TB HDD   

                               gpu  
laptop_ID                           
1118            AMD FirePro W6150M  
153        Nvidia GeForce GTX 1050  
275        Intel Iris Graphics 550  
1100         Intel HD Graphics 520  
131                 AMD Radeon 530  


In [48]:
print(X_train[['memory']].nunique())
print(X_train['memory'].unique())

memory    33
dtype: int64
['1TB HDD' '512GB SSD' '500GB HDD' '256GB SSD +  2TB HDD' '256GB SSD'
 '128GB SSD' '512GB SSD +  1TB HDD' '256GB SSD +  1TB HDD'
 '128GB SSD +  1TB HDD' '2TB HDD' '1TB SSD' '32GB Flash Storage'
 '64GB Flash Storage' '256GB Flash Storage' '8GB SSD' '1TB SSD +  1TB HDD'
 '16GB Flash Storage' '512GB Flash Storage' '128GB Flash Storage'
 '240GB SSD' '512GB SSD +  2TB HDD' '128GB HDD' '180GB SSD'
 '1TB HDD +  1TB HDD' '512GB SSD +  512GB SSD' '1.0TB Hybrid' '1.0TB HDD'
 '32GB SSD' '256GB SSD +  256GB SSD' '16GB SSD' '128GB SSD +  2TB HDD'
 '508GB Hybrid' '64GB SSD']


In [49]:
X_train = X_train.drop(columns=['product'])
X_test = X_test.drop(columns=['product'])
X_train.columns

Index(['inches', 'cpu', 'memory', 'gpu', 'ram_num', 'weight_num',
       'typename_encoded', 'screenresolution_encoded', 'company_Apple',
       'company_Asus', 'company_Chuwi', 'company_Dell', 'company_Fujitsu',
       'company_Google', 'company_HP', 'company_Lenovo', 'company_MSI',
       'company_Mediacom', 'company_Microsoft', 'company_Razer',
       'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi',
       'opsys_Chrome OS', 'opsys_Linux', 'opsys_Mac OS X', 'opsys_No OS',
       'opsys_Windows 10', 'opsys_Windows 10 S', 'opsys_Windows 7',
       'opsys_macOS'],
      dtype='object')

Todas las columnas a minúsculas.
Ram --> Quitarle GB en una columna nueva y convertir a int.  
Weight --> Quitarle kg en una columna nueva y convertir a float.   
TypeName -->Encoding ordenada. Netbook < Notebook < 2 in 1 Convertible < Ultrabook < Gaming < Workstation.  
ScreenResolution -->Encoding ordenada mediante mapeo.    
Company, opsys --> One-Hot-encoding.    
 Product--> ¿Me la quedo? Tiene alta cardinalidad, posibilidad de convertir en bins según Company. ¿Cómo puedo hacer esto? NO ME LA QUEDO.    
Cpu --> One-Hot-encoding.???   



-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [51]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error

cb = CatBoostRegressor(n_estimators=200,
                      loss_function='RMSE',
                       learning_rate=0.4,
                       random_state=1,
                       verbose = False
                      )
pool_train = Pool(X_train, y_train,
                 cat_features=['cpu', 'memory', 'gpu'])

pool_test = Pool(X_test, cat_features=['cpu', 'memory', 'gpu'])

cb.fit(pool_train)
y_pred = cb.predict(pool_test)

print('RMSE:', np.sqrt(mean_squared_error(y_test,y_pred)))

RMSE: 288.51002912197947


In [52]:
df[target_str].describe()

count     912.000000
mean     1111.724090
std       687.959172
min       174.000000
25%       589.000000
50%       978.000000
75%      1483.942500
max      6099.000000
Name: price_in_euros, dtype: float64

### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [53]:
X_pred = pd.read_csv("./data/test.csv", index_col= "laptop_ID")
X_pred.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [54]:
X_pred.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [55]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
Index: 391 entries, 209 to 421
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           391 non-null    object 
 1   Product           391 non-null    object 
 2   TypeName          391 non-null    object 
 3   Inches            391 non-null    float64
 4   ScreenResolution  391 non-null    object 
 5   Cpu               391 non-null    object 
 6   Ram               391 non-null    object 
 7   Memory            391 non-null    object 
 8   Gpu               391 non-null    object 
 9   OpSys             391 non-null    object 
 10  Weight            391 non-null    object 
dtypes: float64(1), object(10)
memory usage: 36.7+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [56]:
X_pred
X_pred.columns = X_pred.columns.str.lower()

X_pred["ram_num"]= X_pred["ram"].str.replace("GB", "")
X_pred["ram_num"] = X_pred["ram_num"].astype(int)
X_pred["weight"]= X_pred["weight"].str.lower()
X_pred["weight_num"]= X_pred["weight"].str.replace("kg", "")
X_pred["weight_num"] = X_pred["weight_num"].astype(float)
X_pred["screenresolution"] = X_pred["screenresolution"].str.lower()

X_pred["typename_encoded"] = ordinal_encoder.transform(X_pred[["typename"]])

X_pred["screenresolution_encoded"] = X_pred["screenresolution"].map(mapeo)

encoded_test = encoder_company.transform(X_pred[['company']])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder_company.get_feature_names_out(), index=X_pred.index)
X_pred = pd.concat([X_pred, encoded_test_df], axis=1)

encoded_test = encoder_opsys.transform(X_pred[['opsys']])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder_opsys.get_feature_names_out(), index=X_pred.index)
X_pred = pd.concat([X_pred, encoded_test_df], axis=1)

c:\Users\Arrate\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [57]:
features_quitar = ["ram"]
features_quitar.append("weight")
features_quitar.append("typename")
features_quitar.append("screenresolution")
features_quitar.append("company")
features_quitar.append("opsys")
features_quitar
X_pred = X_pred.drop(columns=features_quitar)
X_pred= X_pred.drop(columns=['product'])

In [ ]:
# Columnas que tiene el modelo
print(cb.feature_names_)

# Columnas que tiene dataset de predicción
print(X_pred.columns.tolist())

# Diferencias
set(cb.feature_names_) - set(X_pred.columns)  # faltan en predicción
set(X_pred.columns) - set(cb.feature_names_)  # sobran en predicción

['inches', 'cpu', 'memory', 'gpu', 'ram_num', 'weight_num', 'typename_encoded', 'screenresolution_encoded', 'company_Apple', 'company_Asus', 'company_Chuwi', 'company_Dell', 'company_Fujitsu', 'company_Google', 'company_HP', 'company_Lenovo', 'company_MSI', 'company_Mediacom', 'company_Microsoft', 'company_Razer', 'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi', 'opsys_Chrome OS', 'opsys_Linux', 'opsys_Mac OS X', 'opsys_No OS', 'opsys_Windows 10', 'opsys_Windows 10 S', 'opsys_Windows 7', 'opsys_macOS']
['inches', 'cpu', 'memory', 'gpu', 'ram_num', 'weight_num', 'typename_encoded', 'screenresolution_encoded', 'company_Apple', 'company_Asus', 'company_Chuwi', 'company_Dell', 'company_Fujitsu', 'company_Google', 'company_HP', 'company_Lenovo', 'company_MSI', 'company_Mediacom', 'company_Microsoft', 'company_Razer', 'company_Samsung', 'company_Toshiba', 'company_Vero', 'company_Xiaomi', 'opsys_Chrome OS', 'opsys_Linux', 'opsys_Mac OS X', 'opsys_No OS', 'opsys_Windows

set()

In [59]:
predictions_submit = cb.predict(X_pred)
predictions_submit

array([1153.38925288,  285.76465617,  427.34377661,  850.19719654,
       1018.20926039,  415.7549094 ,  730.70644484,  898.0400859 ,
       1109.0335041 ,  295.86261635, 2247.64895928, 1480.9064056 ,
        558.18898475, 1773.37000937,  849.50814024,  726.60761377,
       2420.23243642, 1404.27302611, 2035.81090328,  662.42496065,
       1775.15886376,  275.14883035,  867.37831828, 1308.98090061,
        428.88257779,  716.93129644,  609.78595994,  865.85654611,
       2739.90326122, 1030.80703212, 2392.98181988,  441.30288324,
        749.30508446, 2902.52319461, 2050.40983077, 1337.96937066,
        596.98838704, 1690.24850755,  931.82183376, 1636.02113817,
        714.66977738,  708.38557977,  486.78212488, 1238.67843381,
       1522.21822804, 1059.71448441, 1032.58887822,  513.71481315,
        862.86064528,  388.26774689, 1820.91077635,  822.46193159,
       1098.73072804,  495.12827835, 1987.89308685, 1607.6102022 ,
        809.18123972,  918.35444756,  851.44097822,  821.43709

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [60]:
# ¿Qué opináis?
# ¿Sí, no? Si

In [61]:
X_pred["Price_in_euros"] = predictions_submit
X_pred = X_pred["Price_in_euros"].copy()
X_pred.head()

laptop_ID
209     1153.389253
1281     285.764656
1168     427.343777
1231     850.197197
1020    1018.209260
Name: Price_in_euros, dtype: float64

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [62]:
sample = pd.read_csv("data/sample_submission.csv")

In [63]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [64]:
sample.shape

(391, 2)

In [65]:
sample.columns

Index(['laptop_ID', 'Price_in_euros'], dtype='object')

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [66]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({
    'laptop_ID': X_pred.index,
    'price_in_euros': predictions_submit
}).reset_index(drop=True)

submission

,laptop_ID,price_in_euros
0,209,1153.389253
1,1281,285.764656
2,1168,427.343777
3,1231,850.197197
4,1020,1018.209260
...,...,...
386,820,2272.983646
387,948,1022.208677
388,483,1934.802806
389,1017,758.096913


In [67]:
submission.head()

,laptop_ID,price_in_euros
0,209,1153.389253
1,1281,285.764656
2,1168,427.343777
3,1231,850.197197
4,1020,1018.209260


In [68]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [69]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [70]:
chequeador(submission)

You're ready to submit!
